# Assignment 09

**Specifications and requirements** for each assignment include compliance with the [Programmer's Pact](https://github.com/satyakisikdar/COMP-271-F26/blob/main/programmers-pact.md).

This assignment comprises the following problems, all on linked lists:

* A Singly Linked List That Checks Itself
* Two Riders
* Clearing the Loop
* A Doubly Linked List

## A warm-up for Quiz 2

Quiz 2 is **in class on Tue Nov 3**, and linked lists are on it. On paper you will not run anything: you will read code that rides a line and say what it prints, fill in the one line that rewires a link, or say what goes wrong when two lines are swapped. Each problem below ends with a short **Trace it before you run it** prompt. It is not graded, and it is the best quiz practice here: draw the boxes and arrows on paper first, then run the code and compare.

## What to submit

A single Python file called `week09.py`.

**This assignment starts from a template, not a blank file.** Download [`week09.py`](./week09.py) and keep it in a folder of its own. The template holds five classes: `Node`, `LinkedList`, `CircularLinkedList`, `DoubleNode` and `DoublyLinkedList`, most of them already written. Your job is to **fill in every method whose body is a single line reading** `raise NotImplementedError(...)`. There are 6 of them, spread across the four problems.

The placeholder line you are replacing looks like this:

```python
raise NotImplementedError("replace this line with your implementation")
```

**Any method still holding a `NotImplementedError` counts as unanswered** and will fail its tests. The placeholder exists so that an unfinished file still runs, not as something to leave behind.

Do **not** rename anything, change any signature, or modify the code that is already written. The autograder calls the template's names exactly as given.

Your solutions should be based on material covered so far in this course, through the Unit Testing lecture on Tue Oct 20. You do not need stacks or queues.

## Raise for the caller, assert for yourself

The Unit Testing deck (Week 9-1) split errors into two kinds, and this assignment grades both:

- **`raise` when the caller made the mistake**: the value came from outside, like an index or a `k`. Each problem says exactly which exception to raise and when. Raise it **before** you change anything, so a refused call leaves the list exactly as it was.
- **`assert` when only your own code can be wrong.** Your list keeps a promise between any two calls (the deck called it an **invariant**): the chain of nodes and the `_size` counter agree. `_check_invariants` is the method that checks that promise. You write it for `LinkedList`; the template gives it for the other two classes, and your methods call it there too.

Every method you write that **changes** a list (`remove_at`, `eliminate_every`, `reverse` and `remove_last`) must do two things. They are worth 0.5 points per method (2 in all), and a method earns its 0.5 only when it does both:

1. **Call `self._check_invariants()`** after it changes the list. Calling it once is enough to earn the point; `eliminate_every` is easiest to debug if it calls it after every removal, so a broken loop is caught in the round that broke it.
2. **Contain at least one `assert`, with a message, that checks the method's own promise**: what its docstring lists under *Postcondition*. The deck's `remove_at` ended with `assert self.size() == old_size - 1, 'size off'`. Where you put it is up to you. One way to state each promise (yours may differ):

| Method | Promise in its docstring | One assert that states it |
| --- | --- | --- |
| `remove_at` | the returned name was the one at `index` | save `expected = self.get(index)` first, then `assert removed.data == expected, "removed the wrong station"` |
| `eliminate_every` | every station was removed exactly once | save `old_size` first, then `assert len(removed) == old_size, "a station was skipped or repeated"` |
| `reverse` | the old first station is now the last | save `old_first = self._head` first, then `assert self._tail is old_first, "..."` |
| `remove_last` | the new tail links to nothing after it | `assert self._tail is None or self._tail.next is None, "..."` |

The one form that does **not** count is the bare `assert something is not None, "..."`. It is useful (see the note on type annotations below), but it only says a node exists, not what the method did. Anything else with a message counts, including `assert self._tail is None, "the loop is not empty"`.

**Never run your file with `python -O`.** That flag deletes every `assert` before the code runs, including yours, so a broken list would sail through silently.

### One change from lecture: removing from an empty line raises

In the Linked Lists II deck (Week 7-2), `remove_first` and `remove_last` returned `None` on an empty line, and their return type was `str | None`. In this assignment, `remove_at` and `remove_last` **raise `IndexError`** instead, and they return a plain `str`. This is what Python's own `list.pop()` does on an empty list, and it is the same choice Thursday's Stacks and Queues I lecture (Week 9-2) made for `Stack.pop`: an empty pop must fail clearly, because a caller who forgets to check for `None` gets a quiet wrong answer instead of an error.

## Three notes on the template

### Why some annotations are in quotes

Look at `Node` in the template:

```python
class Node:
    def __init__(self, data: str) -> None:
        self._next: Node | None = None       # no quotes needed here

    @property
    def next(self) -> "Node | None":         # quotes needed here
        return self._next
```

The difference is **when Python reads the annotation**. Python builds a class by running its body from top to bottom, and the name `Node` only exists once that body has finished. A `def` line runs while the body is still running, so Python evaluates the return annotation on it right then, before `Node` exists. Written without quotes, `-> Node | None` stops the program with `NameError: name 'Node' is not defined`. In quotes, it is just a string, and Python stores it without evaluating it. The type checker still reads it as the type `Node | None`.

The annotation on `self._next` inside `__init__` needs no quotes because Python never evaluates an annotation on an attribute inside a method at all. Only the type checker reads it.

This is the same lesson as the midterm's question on annotations, from the other side: Python needs every **name** in an annotation to exist when that line runs, but it never checks the **types** at run time. Checking types is the type checker's job.

### Asserts that tell the type checker a node exists

The type checker knows that `current.next` has type `Node | None`. So after `current = current.next`, it will not let you write `current.data`: `current` might be `None`, and `None` has no `data`. If you know `current` cannot be `None` at that point (because `_size` says the station exists, for example), say so with an assert:

```python
assert current is not None, "ran off the end of the line"
current = current.next
```

After that line, the type checker treats `current` as a `Node`. The template's `get` does exactly this, as did `get` on the Week 9-1 deck's "assert is for bugs, raise is for callers" slide. Your submission's annotations are checked by the type checker, so a message about `None` in that check usually means one of these asserts is missing.

You can see these messages before you submit. In VS Code, a red squiggle under `current.data` saying that `data` is not a known attribute of `None` is the same problem: add the assert above it. Local variables such as `current` or `before` need no annotations of their own; only the template's signatures do.

### One more thing the template enforces: names never move

`Node` and `DoubleNode` give `data` a getter but **no setter**, so `node.data = "Kedzie"` fails with an error. Every problem that rearranges a line (`eliminate_every`, `reverse`) must do it by rewiring links, never by moving names from one node to another.

## Rules for every method you write

- **One `return` statement** in a method that returns a value, and **none** in one that returns nothing. Set a variable in each branch and return it once at the end.
- **No `break` or `continue`.** Put the stopping condition in the `while` test instead, the way the given `CircularLinkedList._check_invariants` keeps riding `while current is not None and current is not self._tail and count <= self._size`.
- **No magic numbers.** `0`, `1` and `-1` may be written bare; any other number needs a named line of its own, such as `STEP = 2`.
- **At least one comment inside each method you write.** The template's own comments do not count toward yours.
- **No `import` statements** anywhere in `week09.py`.
- Use **sensible type annotations**. The template's signatures are already annotated; keep them that way.

## Generative AI policy

You may use generative AI tools to get hints or to clarify concepts, provided you **cite** the tool and how you used it (a short comment at the top of `week09.py` is sufficient, e.g. `# Used ChatGPT to clarify why a circular list keeps _tail instead of _head`). You may **not** use generative AI to generate a solution to any of these problems verbatim: the code you submit must be your own.

### Turn off AI assistance in your editor before you start

**This is required, not a suggestion.** Autocomplete that writes the next line for you is generative AI, even when it does not feel like it. An inline suggestion that writes your unlinking line while you are still drawing the boxes and arrows is the same thing as pasting in a generated solution, and it defeats the point of the assignment. Turn these features off before you open `week09.py`, and leave them off until you submit.

**The reliable way: disable the extensions.** In VS Code, open the Extensions view (`Cmd+Shift+X` on macOS, `Ctrl+Shift+X` on Windows/Linux), search for each of the following, and click **Disable**:

- **GitHub Copilot**
- **GitHub Copilot Chat**
- any other AI assistant you have installed (Codeium / Windsurf, Tabnine, Amazon Q, Continue, Cline, and so on)

Then open the Command Palette (`Cmd+Shift+P` / `Ctrl+Shift+P`), run **GitHub Copilot: Sign Out** if it appears, and check that the Copilot icon in the status bar is gone or shows as disabled.

**Also switch off generic inline suggestions.** Open Settings (`Cmd+,` / `Ctrl+,`), search for `inline suggest`, and uncheck **Editor > Inline Suggest: Enabled**. Equivalently, add this to your `settings.json`:

```json
{
    "github.copilot.enable": { "*": false },
    "editor.inlineSuggest.enabled": false
}
```

Setting names move around between VS Code releases, so if you cannot find one of these, disabling the extensions above is what actually matters.

**Using a different editor?** The same rule applies. Cursor and Windsurf are AI editors: turn off Tab autocomplete and the chat sidebar, or just use plain VS Code for this assignment. In PyCharm, disable **JetBrains AI Assistant** and **Full Line Code Completion**.

**Check that it worked.** Open `week09.py`, put your cursor on a blank line inside one of the unimplemented functions, and wait a few seconds. If grayed-out code appears ahead of your cursor, something is still switched on.

Working without these tools is the point. Quiz 2 is four days after this is due, and it will not have autocomplete either.

---

## Problem 1: A Singly Linked List That Checks Itself

The template's `LinkedList` is the Brown Line version of the lecture's: `Node`s chained from `_head`, plus a counter `_size` that every change keeps up to date, so `size()` answers with no ride. The given `add_first`, `add_last` and `get` all keep `_size` right. The Unit Testing deck (Week 9-1) built `remove_at` on exactly this list and showed what happens when one line of it is wrong. Here you write it yourself, with its safety net.

Implement two methods:

- `_check_invariants()` rides the whole line, counts the nodes, and asserts that the count equals `_size`. It returns nothing when the line is healthy, and raises `AssertionError`, with a message, when it is not. It is the first method because `remove_at` calls it.
- `remove_at(index)` removes the station at position `index` (the head is index 0) and returns its name. As the Linked Lists III deck (Week 8-2) put it, `remove_at` rides to `index - 1`, like `add_after` and `remove`: it stops at the station **before** the one it changes. Removing the head is the one case with no station before it.

### Example

```python
line = brown_line()          # Kimball -> Kedzie -> Francisco -> Rockwell (see the template's __main__)
line._check_invariants()     # a healthy line: nothing happens
line.remove_at(2)            -> 'Francisco'
print(line)                  # Kimball -> Kedzie -> Rockwell
line.remove_at(0)            -> 'Kimball'
line.remove_at(1)            -> 'Rockwell'
print(line, line.size())     # Kedzie 1
```

**Explanation:** after `remove_at(2)`, Kedzie links straight to Rockwell. `remove_at(0)` moves `_head` one stop down the line. By the time `remove_at(1)` runs, only Kedzie and Rockwell are left, so index 1 is Rockwell.

### Example

```python
line.remove_at(1)            -> IndexError    (line still: Kedzie)
line.remove_at(-1)           -> IndexError
LinkedList().remove_at(0)    -> IndexError
line._size = 7
line._check_invariants()     -> AssertionError
```

**Explanation:** a one-station line has only index 0, so asking for index 1 is the caller's mistake: `remove_at` raises before changing anything. A negative index is refused too, even though Python lists accept one. Setting `_size` from outside the class (which real code must never do) breaks the promise, and `_check_invariants` catches it.

### Trace it before you run it

Draw the Brown Line as boxes and arrows. For `remove_at(2)`, mark where your `previous` rider stops, then draw the one arrow that changes. Then do the same for `remove_at(0)`: why is there no `previous` this time?

## Constraints

- `_check_invariants` returns nothing, so it has **no** `return` statement. When the count and `_size` disagree it fails an `assert` with a message; it never raises any other exception for that.
- `remove_at` raises `IndexError` unless `0 <= index < self.size()`, including for every index on an empty line, and leaves the line unchanged when it does.
- `remove_at` calls `self._check_invariants()` after the change and contains a postcondition `assert` with a message (see "Raise for the caller, assert for yourself" above).
- Tests use lines of at most **150** stations.
- The [Programmer's Pact](https://github.com/satyakisikdar/COMP-271-F26/blob/main/programmers-pact.md) applies to everything you write.

## What to Submit

Implement these two methods of `LinkedList` in `week09.py`, keeping the signatures exactly as the template gives them:

| Method | Returns |
| --- | --- |
| `_check_invariants(self) -> None` | nothing; fails an assert if the count and `_size` disagree |
| `remove_at(self, index: int) -> str` | the name of the station that was at `index` |

---

## Problem 2: Two Riders

The Linked Lists III deck (Week 8-2) found the middle of a line in one ride with two riders, `slow` and `fast`, and ended with one more use of the same trick: *if `fast` ever catches up with `slow`, the line has a loop*. This problem asks you to write it.

`has_loop()` returns `True` if following `next` from the head ever comes back to a node it has already visited, and `False` if the ride reaches `None`. Move `slow` one stop and `fast` two stops per turn. If there is no loop, `fast` falls off the end. If there is a loop, `fast` can never fall off; it laps `slow`, and the two land on the same node.

You may not count the line: **no `_size`, no `size()`, no `len()`**. A line with a loop has no end to count to, and `_size` would not tell you whether some link points backward.

### Example

```python
line = brown_line()          # Kimball -> Kedzie -> Francisco -> Rockwell
line.has_loop()              -> False
LinkedList().has_loop()      -> False
# link Rockwell back to Kedzie, on purpose:
line.has_loop()              -> True
```

**Explanation:** after the last station links back to the second, following `next` goes Kimball, Kedzie, Francisco, Rockwell, Kedzie, Francisco, ... forever. Never `print` a line like that: `__str__` rides until `None`, and there is no `None` to reach.

### Trace it before you run it

Both riders start at the head, as in the deck's `middle`. On the looped line above, write where `slow` and `fast` stand after each turn until they meet. How many turns does it take? Then try a loop from Rockwell back to Kimball.

## Constraints

- `has_loop` may not read `_size`, call `size()`, or call `len()`, and may not change any link.
- Move one rider one stop and the other two stops per turn. Watch the guard, as the deck did: `fast.next.next` only exists if `fast.next` does.
- Tests use lines of at most **200** stations, with loops back to every possible station.
- The [Programmer's Pact](https://github.com/satyakisikdar/COMP-271-F26/blob/main/programmers-pact.md) applies to everything you write.

## What to Submit

Implement this method of `LinkedList` in `week09.py`:

| Method | Returns |
| --- | --- |
| `has_loop(self) -> bool` | whether following `next` from the head ever revisits a node |

---

## Problem 3: Clearing the Loop

The template's `CircularLinkedList` is the Linked Lists III deck's (Week 8-2), running round the downtown Loop. The last station links back to the first, and the list keeps `_tail` rather than `_head`: the first station is `_tail.next`, so one reference reaches both ends. Its `_check_invariants` is given.

The deck's table of differences named a circular list's typical use: **round-robin turns**. Write `eliminate_every(k)`, which empties the loop round-robin and returns the names in the order they were removed:

- Start counting at the first station, which counts as 1.
- Remove the `k`-th station counted.
- Start counting again at the station after the one you removed, going round the loop as many times as needed.
- Stop when no station is left.

With `k == 1` the stations leave in order. Use the deck's trick for removing from a loop, "start before the head": in a circular list that means **start at `_tail`**, the station before the first. A pointer that sits on the station *before* the one you are counting can unlink the next one with a single assignment.

### Example

```python
loop = the_loop()            # built with add_last in the template's __main__:
                             # Clark/Lake -> State/Lake -> Washington/Wabash -> Adams/Wabash -> Quincy
loop.eliminate_every(2)
    -> ['State/Lake', 'Adams/Wabash', 'Clark/Lake', 'Quincy', 'Washington/Wabash']
loop.is_empty(), loop.size(), str(loop)    -> True 0 ''
```

**Explanation:** counting 1, 2 removes State/Lake. Counting starts again at Washington/Wabash: 1, then Adams/Wabash is 2 and leaves. Then Quincy is 1 and Clark/Lake is 2 (the count went round the end of the loop). Then Washington/Wabash is 1 and Quincy is 2, leaving Washington/Wabash alone; it counts as both 1 and 2 and leaves last.

### Example

```python
the_loop().eliminate_every(1)
    -> ['Clark/Lake', 'State/Lake', 'Washington/Wabash', 'Adams/Wabash', 'Quincy']
the_loop().eliminate_every(7)
    -> ['State/Lake', 'Quincy', 'Clark/Lake', 'Washington/Wabash', 'Adams/Wabash']
CircularLinkedList().eliminate_every(3)    -> []
the_loop().eliminate_every(0)              -> ValueError
```

**Explanation:** `k` may be larger than the loop: counting 7 round a 5-station loop goes round once and stops at the second station, State/Lake. An empty loop has nothing to remove. A `k` below 1 names no station at all.

### Trace it before you run it

For `eliminate_every(2)` on the Loop, draw the stations in a circle and write, for each round, where your `before` pointer stands and which arrow you redraw. In which round does the station you remove happen to be `_tail`, and what has to happen to `_tail` then?

## Constraints

- Raise `ValueError` if `k < 1`, before changing anything. An empty loop returns `[]`.
- Unlink each station by rewiring the station before it. **Do not create nodes**, do not call `add_first`, `add_last` or any other method that changes the loop, and do not work the order out on a Python list first (no `.pop()`, `.insert()`, `.remove()` or `.index()`). The list you return just collects the names as stations leave.
- Two cases need care, as on the deck's removal slide: removing the only station left empties the loop (`_tail` becomes `None`), and removing the tail moves `_tail` back to the station before it.
- Call `self._check_invariants()` after each removal, and contain a postcondition `assert` with a message.
- When it returns, the loop is empty: `_tail` is `None` and `size()` is `0`.
- Tests use loops of at most **200** stations and `k` up to about 25.
- The [Programmer's Pact](https://github.com/satyakisikdar/COMP-271-F26/blob/main/programmers-pact.md) applies to everything you write.

## What to Submit

Implement this method of `CircularLinkedList` in `week09.py`:

| Method | Returns |
| --- | --- |
| `eliminate_every(self, k: int) -> list[str]` | the station names in the order they were removed |

---

## Problem 4: A Doubly Linked List

The template's `DoublyLinkedList` is the Orange Line version of the Linked Lists III deck's (Week 8-2): each `DoubleNode` links to the station after it (`next`) and the one before it (`prev`), and the list keeps both `_head` and `_tail`. Twice as many links means twice as many ways to get one wrong, so the template gives this class a `_check_invariants` of its own. It rides from `_head` and asserts that each node's `prev` is the node the ride just left, that the ride ends on `_tail`, and that the count equals `_size`. Read it before you start: it is the list of promises your two methods must keep.

Implement two methods:

- `reverse()` reverses the line **in place**: the old tail becomes the head. The deck's singly linked `reverse` needed three riders because each node knew only the way forward. Here each node already knows both neighbors, so reversing means **swapping each node's `next` and `prev`**, then swapping `_head` and `_tail`. Save the way on before you swap, just as `following` did in the deck.
- `remove_last()` removes the last station and returns its name. The deck's table showed why a doubly linked list exists: **the tail already knows the station before it**, so there is no ride to the end. An empty line raises `IndexError` (see "One change from lecture" above).

### Example

```python
orange = DoublyLinkedList()  # then add_last for each of:
                             # Midway -> Pulaski -> Kedzie -> Western -> 35th/Archer
orange._check_invariants()   # a healthy line: nothing happens
orange.remove_last()         -> '35th/Archer'
print(orange)                # Midway -> Pulaski -> Kedzie -> Western
orange.reverse()
print(orange)                # Western -> Kedzie -> Pulaski -> Midway
orange.remove_last()         -> 'Midway'
print(orange)                # Western -> Kedzie -> Pulaski
orange.reverse()
orange.add_last("Halsted")
print(orange)                # Pulaski -> Kedzie -> Western -> Halsted
```

**Explanation:** after `reverse()`, the old head Midway is the tail, so the next `remove_last()` removes it. Reversing back and calling the given `add_last` works only if every `prev` link was swapped too: `add_last` relies on `_tail` and its links being right.

### Example

```python
# after removing all four stations:
orange.reverse()             # an empty line stays empty
orange.remove_last()         -> IndexError
# with Midway -> Pulaski -> Kedzie, cut one backward link on purpose (the tail's prev set to the head):
orange._check_invariants()   -> AssertionError
```

**Explanation:** Kedzie's `prev` now points at Midway instead of Pulaski. Riding forward still visits all three stations, and `_size` is still 3, so only the given check's `prev` test catches it. A `reverse` that forgets to swap one `prev` is caught the same way.

### Trace it before you run it

Draw the four-station Orange Line with both arrows between each pair of stations. For `reverse()`, redraw the arrows at each station in turn: which arrows have been swapped after the first station, and where does your rider go next? For `remove_last()`, how many arrows change?

## Constraints

- `reverse` creates **no nodes** and builds **no list** (no collecting the nodes or names first); it swaps links as it rides. An empty line stays empty.
- `remove_last` has **no loop** of its own. It raises `IndexError` on an empty line, before changing anything. Removing the only station leaves both `_head` and `_tail` as `None`; otherwise the new tail links to nothing after it. The removed node's own links do not matter once nothing points to it.
- `reverse` and `remove_last` each call `self._check_invariants()` after the change and contain a postcondition `assert` with a message. (That call does ride the line; that is the price of checking, and it is not the method's own loop.)
- Tests use lines of at most **10** stations.
- The [Programmer's Pact](https://github.com/satyakisikdar/COMP-271-F26/blob/main/programmers-pact.md) applies to everything you write.

## What to Submit

Implement these two methods of `DoublyLinkedList` in `week09.py`:

| Method | Returns |
| --- | --- |
| `reverse(self) -> None` | nothing; reverses the line in place |
| `remove_last(self) -> str` | the name of the station that was last |

---

## Submitting on Gradescope

- Download the template [`week09.py`](./week09.py), fill in the six methods marked with `NotImplementedError`, and submit **that file** to the **Week09** assignment on Gradescope. Keep the name `week09.py`. (If you submit a single `.py` file under another name, it is still graded, and the feedback says so; a notebook or a `.zip` is not.)
- Don't submit this notebook. The notebook is the instructions; `week09.py` is the answer.
- Run `uv run week09.py` locally before you submit. The template's `if __name__ == "__main__":` block prints each result next to the value it should have, and every line should match. Each problem's checks run on their own, so you can do the problems in any order: an unfinished problem prints `problem_3: not finished yet, skipped` (with its own number), and one that crashes prints the error and lets the others run. A link rewired into a loop does not crash: the ride just never ends, and the program sits there printing nothing. If a run does not finish within a few seconds, press **Ctrl+C** to stop it, then look for a link that points back at a station the ride has already passed.
- **Released Fri Oct 23. Due Fri Oct 30 at 11:55 pm.** You may resubmit as many times as you like before the deadline; only your last submission counts.
- **Late pass:** if you use one, the deadline is **Mon Nov 2 at 11:55 pm**.
- **Solutions** go out after the late-pass deadline (Mon Nov 2), in time to study before Quiz 2 on Tue Nov 3.
- Before submitting, read your code against the [Programmer's Pact](https://github.com/satyakisikdar/COMP-271-F26/blob/main/programmers-pact.md) and each problem's constraints, the same way the autograder will.
- The score is out of 30: 24 for correctness, 2 for the asserts described in "Raise for the caller, assert for yourself", and 4 for the Pact and each problem's constraints.
- Feedback comes in two tiers:
    - **Visible immediately** on every submission: the examples from this notebook and the errors each problem's constraints state, the two assert checks for each method that changes a list, and every Pact and constraint check. These tell you right away if something structural is wrong.
    - **Released after grades are published**: a broader sweep of edge cases (every index, every loop shape, every size of loop with every `k` up to 10), with more detailed feedback, including a pointer to where in your code a failure happened. These count toward your score the whole time; you see their results once grades are out.
- A method whose ride never ends is stopped after 5 seconds, and the feedback says so. That almost always means a link was rewired into a loop.
- If the autograder's output doesn't make sense to you, or you think it's wrong, ask on the course forum or come to office hours rather than guessing repeatedly.